# 환각의 구조적 원인

모델이 알지 못하는 고유 사실을 물었을 때, 왜 그럴듯한 문장을 이어 말할 수 있을까요?

Python 3.12와 잠긴 패키지를 사용합니다. 첫 실행에서는 로컬 모델을 다운로드합니다.

## 로컬 생성 모델 준비

In [1]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, logging, set_seed

logging.set_verbosity_error()

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION)
model.eval()
print("model:", MODEL_ID)
print("parameters:", f"{model.num_parameters():,}")

model: Qwen/Qwen2.5-0.5B-Instruct
parameters: 494,032,768


In [2]:
def generate(prompt: str, *, temperature: float = 0.7, seed: int = 0, max_new_tokens: int = 48) -> str:
    # 채팅 템플릿을 적용한 뒤 새로 생성된 토큰만 복원합니다.
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")
    set_seed(seed)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=0.9 if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 근거가 없는 고유 정보 질문

실습을 위해 만든 기관과 정책을 묻습니다. 질문 문장에는 정답 근거가 없습니다.

In [3]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
questions = [
    "2026년 가상 기관 '해바라기 연구소'의 초록 패스 제도 지원금은 얼마인가요?",
    "가상 제품 '노을 워커 3'의 배터리 용량은 몇 mAh인가요?",
]
without_context = []
for question in questions:
    print("Q:", question)
    answer = generate(question, temperature=0, max_new_tokens=64)
    without_context.append(answer)
    print("A:", answer)

Q: 2026년 가상 기관 '해바라기 연구소'의 초록 패스 제도 지원금은 얼마인가요?


A: 해바라기 연구소의 초록 패스 제도 지원금은 2026년에 결정되었습니다. 이는 해바라기 연구소가 학술적으로 인정받고 있는 연구를 위한 지원을 제공하는 방식으로, 이는 연구 결과를 공개하고 교
Q: 가상 제품 '노을 워커 3'의 배터리 용량은 몇 mAh인가요?


A: "노을 워커 3"라는 이름으로 인해, 실제 제품에 대한 정보는 없습니다. "노을 워커 3"이라는 제품이 실제로 존재하지 않는 것으로 보입니다. 이 제품의 배터리 용량에 대해 구체적인 정보를 제공하려면,


## 답변 제약을 추가한 비교

정보가 주어지지 않으면 알 수 없다고 답하도록 제약을 추가합니다. 이 제약이 항상 정확한 답을 보장하지는 않습니다.

In [4]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
instruction = "주어진 질문에 정답 근거가 없으면 추측하지 말고 '제공된 정보로는 확인할 수 없습니다'라고 답하세요.\n\n"
with_constraint = []
for question in questions:
    print("Q:", question)
    answer = generate(instruction + question, temperature=0, max_new_tokens=64)
    with_constraint.append(answer)
    print("A:", answer)

Q: 2026년 가상 기관 '해바라기 연구소'의 초록 패스 제도 지원금은 얼마인가요?


A: 제공된 정보로는 확인할 수 없습니다.
Q: 가상 제품 '노을 워커 3'의 배터리 용량은 몇 mAh인가요?


A: 제공된 정보로는 확인할 수 없습니다.


## 구조 검증

예상한 실험 조건과 출력 구조가 유지되었는지 확인합니다.

In [5]:
assert len(without_context) == 2 and len(with_constraint) == 2
print("검증: 근거 제약 전·후의 응답을 각각 2개 확인했습니다.")

검증: 근거 제약 전·후의 응답을 각각 2개 확인했습니다.
